In [1]:
!pip install node2vec
!pip install line-embeddings
!pip install torch-geometric torch-scatter torch-sparse -q


ERROR: Could not find a version that satisfies the requirement line-embeddings (from versions: none)
ERROR: No matching distribution found for line-embeddings
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.8 MB/s eta 0:00:00


In [2]:
# %% Cell: Imports & utilities
import os
import random
import math
from collections import Counter
from pprint import pprint


import numpy as np
import pandas as pd
import networkx as nx
import xml.etree.ElementTree as ET
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score


# torch + pyg
import torch
import torch.nn.functional as F
from torch_geometric.utils import from_networkx, to_undirected, negative_sampling
from torch_geometric.loader import NeighborSampler, NeighborLoader
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, GAE, VGAE


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
import networkx as nx
import pandas as pd
import xml.etree.ElementTree as ET

def parse_graphml(file_path):
    """Parse GraphML file and return NetworkX graph"""
    try:
        G = nx.read_graphml(file_path)
        return G
    except Exception:
        return parse_graphml_manual(file_path)

def parse_graphml_manual(file_path):
    """Manual fallback parsing of GraphML"""
    tree = ET.parse(file_path)
    root = tree.getroot()
    G = nx.Graph()

    # Parse keys
    keys = {}
    for key_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}key'):
        keys[key_elem.get('id')] = key_elem.get('attr.name')

    # Parse nodes
    for node_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}node'):
        G.add_node(node_elem.get('id'))

    # Parse edges with attributes
    for edge_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}edge'):
        source = edge_elem.get('source')
        target = edge_elem.get('target')
        edge_attrs = {}

        for data_elem in edge_elem.findall('.//{http://graphml.graphdrawing.org/xmlns}data'):
            key_id = data_elem.get('key')
            attr_name = keys.get(key_id)
            if attr_name:
                try:
                    edge_attrs[attr_name] = float(data_elem.text)
                except:
                    edge_attrs[attr_name] = data_elem.text

        G.add_edge(source, target, **edge_attrs)

    return G

def extract_edge_features(G):
    """Extract edge features as pandas DataFrame"""
    rows = []
    for u, v, data in G.edges(data=True):
        row = {"source": u, "target": v}
        row.update(data)
        rows.append(row)
    return pd.DataFrame(rows)


# ------------ MAIN PART (import + exploration) ----------------

file_path = "/kaggle/input/network-intrusion-dataset/0.1M-Stratified-Multi.graphml"

# Load graph
G = parse_graphml(file_path)

print("Number of nodes :", G.number_of_nodes())
print("Number of edges :", G.number_of_edges())

# Extract edge features
edge_df = extract_edge_features(G)

print("\nEdge feature columns :", edge_df.columns.tolist())
print("\nEdge features shape :", edge_df.shape)

print("\nSummary statistics:\n")
print(edge_df.describe())


Number of nodes : 41073
Number of edges : 100000

Edge feature columns : ['source', 'target', 'TotPkts', 'TotBytes', 'SrcBytes', 'Dur', 'Proto_encoded', 'Dir_encoded', 'State_encoded', 'ActivityLabel']

Edge features shape : (100000, 10)

Summary statistics:

             TotPkts       TotBytes       SrcBytes            Dur  \
count  100000.000000  100000.000000  100000.000000  100000.000000   
mean        0.000161      -0.000150       0.000881       0.003185   
std         0.612019       0.745613       0.810429       1.003557   
min        -0.011045      -0.012529      -0.013294      -0.373111   
25%        -0.010884      -0.012494      -0.013254      -0.373111   
50%        -0.010884      -0.012480      -0.013250      -0.373110   
75%        -0.010081      -0.012323      -0.013044      -0.364371   
max       149.951002     183.902708      72.277354       3.705196   

       Proto_encoded    Dir_encoded  State_encoded  ActivityLabel  
count  100000.000000  100000.000000  100000.000000

In [4]:
import networkx as nx

# Supposons que G est le graphe complet
largest_cc = max(nx.connected_components(G), key=len)
G_sub = G.subgraph(largest_cc).copy()

print(f"Nodes: {G_sub.number_of_nodes()}, Edges: {G_sub.number_of_edges()}")


Nodes: 40863, Edges: 99833


In [5]:
import networkx as nx
import random

# ---------------- Load original graph ----------------
G_full = parse_graphml(file_path)
print(f"Original Graph -> Nodes: {G_full.number_of_nodes()} | Edges: {G_full.number_of_edges()}")

# ---------------- Sample edges pour sous-graphe connecté ----------------
target_edges = 12000
G = nx.Graph()

edges = list(G_full.edges())
random.seed(42)
random.shuffle(edges)

for u, v in edges:
    G.add_edge(u, v)
    # Stopper quand on a assez de nodes et d’edges
    if G.number_of_nodes() >= 2500 and G.number_of_edges() >= target_edges:
        break

# ---------------- Vérifier connectivité ----------------
if nx.is_connected(G):
    print("Le sous-graphe est connecté ✅")
else:
    largest_cc = max(nx.connected_components(G), key=len)
    G = G.subgraph(largest_cc).copy()
    print(f"Sous-graphe ajusté -> Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")


Original Graph -> Nodes: 41073 | Edges: 100000
Sous-graphe ajusté -> Nodes: 10650, Edges: 11838


In [6]:
if not nx.is_connected(G_sub):
    # garder seulement la plus grande composante
    largest_cc = max(nx.connected_components(G), key=len)
    G = G_sub.subgraph(largest_cc).copy()


In [7]:
import networkx as nx

# Vérifier si le sous-graphe est connecté
if nx.is_connected(G_sub):
    print("Le sous-graphe est connecté ✅")
else:
    # Prendre la plus grande composante connectée
    largest_cc = max(nx.connected_components(G), key=len)
    G = G.subgraph(largest_cc).copy()
    print(f"Sous-graphe ajusté -> Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")


Le sous-graphe est connecté ✅


# Traditional Feature-based Methods 

In [8]:
import random

def generate_negative_edges(G, num_samples):
    """
    Generate negative edges (u, v) that DO NOT exist in the graph G.
    """
    neg_edges = set()
    nodes = list(G.nodes())
    existing = set(G.edges()) | { (v, u) for (u, v) in G.edges() }

    while len(neg_edges) < num_samples:
        u = random.choice(nodes)
        v = random.choice(nodes)
        if u == v:
            continue
        if (u, v) in existing:
            continue
        neg_edges.add((u, v))

    return list(neg_edges)


In [9]:
import random

def generate_negative_edges(G, num_samples):
    neg_edges = set()
    nodes = list(G.nodes())
    existing = set(G.edges()) | { (v, u) for (u, v) in G.edges() }

    while len(neg_edges) < num_samples:
        u = random.choice(nodes)
        v = random.choice(nodes)
        if u == v or (u, v) in existing:
            continue
        neg_edges.add((u, v))
    
    return list(neg_edges)


def score_heuristic(G, edges, method):
    scores = []
    for u, v in edges:
        if method == "cn":
            s = len(list(nx.common_neighbors(G, u, v)))
        elif method == "jaccard":
            s = list(nx.jaccard_coefficient(G, [(u, v)]))[0][2]
        elif method == "aa":
            s = list(nx.adamic_adar_index(G, [(u, v)]))[0][2]
        elif method == "ra":
            s = list(nx.resource_allocation_index(G, [(u, v)]))[0][2]
        elif method == "pa":
            s = list(nx.preferential_attachment(G, [(u, v)]))[0][2]
        else:
            s = 0
        scores.append(s)
    return scores


methods = {
    "common_neighbors": "cn",
    "jaccard": "jaccard",
    "adamic_adar": "aa",
    "resource_allocation": "ra",
    "preferential_attachment": "pa"
}

results_heuristics = {}

neg_val = generate_negative_edges(G_train, len(val_edges))

for name, m in methods.items():
    pos = score_heuristic(G_train, val_edges, m)
    neg = score_heuristic(G_train, neg_val, m)
    results_heuristics[name] = evaluate(pos, neg)

results_heuristics


NameError: name 'G_train' is not defined

In [ ]:
import numpy as np
import networkx as nx
import random
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score


# ------------------------------
# Negative sampling
# ------------------------------
def generate_negative_edges(G, num_samples):
    neg_edges = set()
    nodes = list(G.nodes())
    existing = set(G.edges()) | {(v, u) for (u, v) in G.edges()}

    while len(neg_edges) < num_samples:
        u = random.choice(nodes)
        v = random.choice(nodes)
        if u == v or (u, v) in existing:
            continue
        neg_edges.add((u, v))

    return list(neg_edges)


# ------------------------------
# Compute heuristic scores
# ------------------------------
def compute_single_heuristic(G, edges, method):

    if method == "cn":
        return [len(list(nx.common_neighbors(G, u, v))) for u, v in edges]

    if method == "jaccard":
        return [s for _, _, s in nx.jaccard_coefficient(G, edges)]

    if method == "aa":
        return [s for _, _, s in nx.adamic_adar_index(G, edges)]

    if method == "ra":
        return [s for _, _, s in nx.resource_allocation_index(G, edges)]

    if method == "pa":
        return [s for _, _, s in nx.preferential_attachment(G, edges)]

    raise ValueError("Unknown heuristic method")


# ------------------------------
# Train ML model on a heuristic
# ------------------------------
def train_ml_on_heuristic(G, train_pos, train_neg, val_pos, val_neg, method):

    # ----- compute train features -----
    X_pos = np.array(compute_single_heuristic(G, train_pos, method)).reshape(-1, 1)
    X_neg = np.array(compute_single_heuristic(G, train_neg, method)).reshape(-1, 1)

    X_train = np.vstack([X_pos, X_neg])
    y_train = np.hstack([np.ones(len(X_pos)), np.zeros(len(X_neg))])

    clf = LogisticRegression(max_iter=300)
    clf.fit(X_train, y_train)

    # ----- compute validation features -----
    X_val_pos = np.array(compute_single_heuristic(G, val_pos, method)).reshape(-1, 1)
    X_val_neg = np.array(compute_single_heuristic(G, val_neg, method)).reshape(-1, 1)

    X_val = np.vstack([X_val_pos, X_val_neg])
    y_val = np.hstack([np.ones(len(X_val_pos)), np.zeros(len(X_val_neg))])

    preds = clf.predict_proba(X_val)[:, 1]

    auc = roc_auc_score(y_val, preds)
    ap = average_precision_score(y_val, preds)

    return auc, ap


# ------------------------------
# Run all heuristic-ML models
# ------------------------------
methods = {
    "common_neighbors": "cn",
    "jaccard": "jaccard",
    "adamic_adar": "aa",
    "resource_allocation": "ra",
    "preferential_attachment": "pa"
}

results = {}

neg_train = generate_negative_edges(G_train, len(train_edges))
neg_val   = generate_negative_edges(G_train, len(val_edges))

for name, code in methods.items():
    auc, ap = train_ml_on_heuristic(
        G_train,
        train_edges,
        neg_train,
        val_edges,
        neg_val,
        code
    )

    results[name] = {"AUC": auc, "AP": ap}
    print(f"{name} + ML → AUC={auc:.4f}  AP={ap:.4f}")

results


In [ ]:
import numpy as np
import networkx as nx
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
    xgb_available = True
except:
    xgb_available = False
    print("⚠ XGBoost n'est pas installé.")

import random
from tqdm import tqdm


In [ ]:
def generate_negative_edges(G, num_samples):
    neg_edges = set()
    nodes = list(G.nodes())
    existing = set(G.edges()) | {(v, u) for (u, v) in G.edges()}

    while len(neg_edges) < num_samples:
        u = random.choice(nodes)
        v = random.choice(nodes)
        if u == v or (u, v) in existing:
            continue
        neg_edges.add((u, v))
    
    return list(neg_edges)


In [ ]:
def compute_heuristic_features(G, edges):
    CN, JC, AA, RA, PA = [], [], [], [], []

    for u, v in edges:
        CN.append(len(list(nx.common_neighbors(G, u, v))))
        JC.append(list(nx.jaccard_coefficient(G, [(u, v)]))[0][2])
        AA.append(list(nx.adamic_adar_index(G, [(u, v)]))[0][2])
        RA.append(list(nx.resource_allocation_index(G, [(u, v)]))[0][2])
        PA.append(list(nx.preferential_attachment(G, [(u, v)]))[0][2])

    return np.vstack([CN, JC, AA, RA, PA]).T  # shape: (n_edges, 5)


In [ ]:
# Exemple : edge_attr_dict[(u,v)] = {"TotBytes":1000, "SrcBytes":40}
edge_attr_dict = {}


In [ ]:
def extract_edge_attr(edges, edge_attr_dict):
    X = []
    for u, v in edges:
        attrs = edge_attr_dict.get((u, v), edge_attr_dict.get((v, u), {}))
        if len(attrs) == 0:
            X.append([0, 0, 0])  # si pas d'infos
        else:
            X.append([
                attrs.get("TotBytes", 0),
                attrs.get("SrcBytes", 0),
                attrs.get("DstBytes", 0)
            ])
    return np.array(X)


In [ ]:
def build_features(G, edges, edge_attr_dict):
    H = compute_heuristic_features(G, edges)
    E = extract_edge_attr(edges, edge_attr_dict)
    return np.hstack([H, E])        # shape = (n_edges, 8)


In [ ]:
def compute_metrics(y_true, scores):
    auc = roc_auc_score(y_true, scores)
    ap = average_precision_score(y_true, scores)

    preds = (scores >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()

    fpr = fp / (fp + tn + 1e-8)
    fnr = fn / (fn + tp + 1e-8)

    return auc, ap, fpr, fnr


In [ ]:
def train_and_eval_model(model, G_train, train_pos, train_neg, val_pos, val_neg, edge_attr_dict):
    X_pos = build_features(G_train, train_pos, edge_attr_dict)
    X_neg = build_features(G_train, train_neg, edge_attr_dict)

    X = np.vstack([X_pos, X_neg])
    y = np.hstack([np.ones(len(X_pos)), np.zeros(len(X_neg))])

    model.fit(X, y)

    # validation
    Xv_pos = build_features(G_train, val_pos, edge_attr_dict)
    Xv_neg = build_features(G_train, val_neg, edge_attr_dict)

    Xv = np.vstack([Xv_pos, Xv_neg])
    yv = np.hstack([np.ones(len(Xv_pos)), np.zeros(len(Xv_neg))])

    scores = model.predict_proba(Xv)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(Xv)

    return compute_metrics(yv, scores)


In [ ]:
results_ml = {}

models = {
    "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=20),
    "SVM-RBF": SVC(kernel="rbf", probability=True)
}

if xgb_available:
    models["XGBoost"] = XGBClassifier(
        max_depth=8,
        n_estimators=300,
        learning_rate=0.05,
        subsample=0.8
    )

# Génération du négatif
train_neg = generate_negative_edges(G_train, len(train_edges))
val_neg   = generate_negative_edges(G_train, len(val_edges))

# Lancer
for name, model in models.items():
    auc, ap, fpr, fnr = train_and_eval_model(
        model, G_train,
        train_edges, train_neg,
        val_edges, val_neg,
        edge_attr_dict
    )

    results_ml[name] = {
        "AUC": auc,
        "AP": ap,
        "FPR": fpr,
        "FNR": fnr
    }


In [ ]:
results_ml


# Shallow embedding 

In [ ]:
from gensim.models import Word2Vec
import numpy as np
import networkx as nx
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from tqdm import tqdm


### DeepWalk : génération des random walks

In [ ]:
def generate_random_walks(G, num_walks=10, walk_length=20):
    walks = []
    nodes = list(G.nodes())

    for _ in range(num_walks):
        random.shuffle(nodes)
        for n in nodes:
            walk = [str(n)]
            cur = n
            for _ in range(walk_length - 1):
                neighbors = list(G.neighbors(cur))
                if len(neighbors) == 0:
                    break
                nxt = random.choice(neighbors)
                walk.append(str(nxt))
                cur = nxt
            walks.append(walk)
    return walks


### Entraînement Word2Vec (DeepWalk / Node2Vec)

In [ ]:
def train_word2vec(walks, dim=128):
    model = Word2Vec(
        sentences=walks,
        vector_size=dim,
        window=5,
        min_count=0,
        sg=1,
        workers=4,
        epochs=5
    )
    return model


### Fonction d’embedding à partir d’un modèle Word2Vec

In [ ]:
def get_embedding(model, node):
    try:
        return model.wv[str(node)]
    except:
        return np.zeros(model.vector_size)


### Link prediction : dot-product score

In [ ]:
def lp_score(model, edges):
    scores = []
    for u, v in edges:
        eu = get_embedding(model, u)
        ev = get_embedding(model, v)
        scores.append(np.dot(eu, ev))
    return np.array(scores)


### Évaluation (AUC, AP, FPR, FNR)

In [ ]:
def compute_metrics(y_true, scores):
    auc = roc_auc_score(y_true, scores)
    ap = average_precision_score(y_true, scores)

    preds = (scores >= np.median(scores)).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()

    fpr = fp / (fp + tn + 1e-8)
    fnr = fn / (fn + tp + 1e-8)

    return auc, ap, fpr, fnr


### DeepWalk

In [ ]:
def run_deepwalk(G_train, train_edges, val_edges, neg_val):
    print("Generating walks for DeepWalk...")
    walks = generate_random_walks(G_train, num_walks=10, walk_length=20)

    print("Training Word2Vec...")
    model = train_word2vec(walks, dim=128)

    pos_scores = lp_score(model, val_edges)
    neg_scores = lp_score(model, neg_val)

    y = np.array([1]*len(pos_scores) + [0]*len(neg_scores))
    s = np.concatenate([pos_scores, neg_scores])

    auc, ap, fpr, fnr = compute_metrics(y, s)

    return auc, ap, fpr, fnr, model  # <-- retourner le modèle aussi


### Node2Vec (random walks biaisés)

In [ ]:
def node2vec_walk(G, start, walk_length, p, q):
    walk = [str(start)]
    cur = start

    for _ in range(walk_length - 1):
        neighbors = list(G.neighbors(cur))
        if len(neighbors) == 0:
            break

        if len(walk) == 1:
            nxt = random.choice(neighbors)
        else:
            prev = walk[-2]
            probs = []
            for n in neighbors:
                if n == prev:
                    probs.append(1/p)
                elif G.has_edge(n, prev):
                    probs.append(1)
                else:
                    probs.append(1/q)
            probs = np.array(probs) / np.sum(probs)
            nxt = np.random.choice(neighbors, p=probs)

        walk.append(str(nxt))
        cur = nxt
    return walk


### Node2Vec complet

In [ ]:
def run_node2vec(G_train, p=0.5, q=2, dim=128):
    print("Generating Node2Vec random walks...")
    walks = []
    for node in tqdm(G_train.nodes()):
        for _ in range(10):
            walks.append(node2vec_walk(G_train, node, walk_length=20, p=p, q=q))

    print("Training Word2Vec...")
    model = train_word2vec(walks, dim=dim)
    return model


### Pipeline global des méthodes shallow embedding

In [ ]:
print("Nodes:", G_train.number_of_nodes())
print("Edges:", G_train.number_of_edges())

if G_train.number_of_edges() == 0:
    raise ValueError("G_train has no edges! LINE cannot run.")


In [ ]:
results_shallow = {}

# --- DeepWalk ---
print("\n=== DeepWalk ===")
auc, ap, fpr, fnr, deepwalk_model = run_deepwalk(G_train, train_edges, val_edges, neg_val)
results_shallow["DeepWalk"] = {"AUC": auc, "AP": ap, "FPR": fpr, "FNR": fnr}

# --- Node2Vec ---
print("\n=== Node2Vec ===")
model = run_node2vec(G_train)
pos = lp_score(model, val_edges)
neg = lp_score(model, neg_val)
y = np.array([1]*len(pos) + [0]*len(neg))
scores = np.concatenate([pos, neg])
auc, ap, fpr, fnr = compute_metrics(y, scores)
results_shallow["Node2Vec"] = {"AUC": auc, "AP": ap, "FPR": fpr, "FNR": fnr}

# --- LINE ---
# Filtrer val_edges pour ne garder que les edges dont les deux nodes sont dans G_train
val_edges_filtered = [(u, v) for u, v in val_edges if u in G_train and v in G_train]
neg_val_filtered   = [(u, v) for u, v in neg_val if u in G_train and v in G_train]



# --- Affichage final ---
results_shallow


### Pipeline complet : Embeddings → Features → ML classifier

Pipeline complet : Embeddings → Features → ML classifier

Fonction générique pour :

récupérer embeddings

construire X (features)

y (labels pos/neg)

entraîner ML

tester sur val ou test

retourner AUC, AP, FPR, FNR

### Feature construction (Hadamard)

In [ ]:
def combine_embeddings(model, edges, dim=128):
    X = []
    for u, v in edges:
        eu = get_embedding(model, u)
        ev = get_embedding(model, v)
        X.append(eu * ev)   # Hadamard product
    return np.array(X)


### Training ML model (Random Forest / XGBoost / SVM)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

def train_ml_classifier(X_train, y_train, model="rf"):
    if model == "rf":
        clf = RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            n_jobs=-1,
            class_weight="balanced"
        )

    elif model == "xgb":
        clf = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            objective="binary:logistic",
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            n_jobs=-1
        )

    elif model == "svm":
        clf = SVC(kernel="rbf", probability=True, class_weight="balanced")

    else:
        raise ValueError("Unknown model")

    clf.fit(X_train, y_train)
    return clf


### Évaluation

In [ ]:
def eval_classifier(clf, X_pos, X_neg):
    y_true = np.array([1]*len(X_pos) + [0]*len(X_neg))

    X = np.vstack([X_pos, X_neg])
    scores = clf.predict_proba(X)[:,1]

    auc = roc_auc_score(y_true, scores)
    ap = average_precision_score(y_true, scores)

    preds = (scores >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()

    fpr = fp / (fp + tn + 1e-8)
    fnr = fn / (fn + tp + 1e-8)

    return {"AUC": auc, "AP": ap, "FPR": fpr, "FNR": fnr}


### Pipeline complet pour une méthode d’embedding (DeepWalk, Node2Vec, SVD, etc.)

In [ ]:
def run_ml_on_embedding(model, train_pos, train_neg, val_pos, val_neg, ml_type="rf"):
    # ---- Build training features ----
    X_pos_train = combine_embeddings(model, train_pos)
    X_neg_train = combine_embeddings(model, train_neg)

    X_train = np.vstack([X_pos_train, X_neg_train])
    y_train = np.array([1]*len(X_pos_train) + [0]*len(X_neg_train))

    # ---- Train ML model ----
    clf = train_ml_classifier(X_train, y_train, model=ml_type)

    # ---- Validation ----
    X_pos_val = combine_embeddings(model, val_pos)
    X_neg_val = combine_embeddings(model, val_neg)

    return eval_classifier(clf, X_pos_val, X_neg_val)


In [ ]:
# --- DeepWalk ---
print("=== DeepWalk ===")
auc, ap, fpr, fnr, deepwalk_model = run_deepwalk(G_train, train_edges, val_edges, neg_val)
results_shallow["DeepWalk"] = {"AUC": auc, "AP": ap, "FPR": fpr, "FNR": fnr}

# --- Node2Vec ---
print("=== Node2Vec ===")
node2vec_model = run_node2vec(G_train)
pos = lp_score(node2vec_model, val_edges)
neg = lp_score(node2vec_model, neg_val)
y = np.array([1]*len(pos) + [0]*len(neg))
scores = np.concatenate([pos, neg])
auc, ap, fpr, fnr = compute_metrics(y, scores)
results_shallow["Node2Vec"] = {"AUC": auc, "AP": ap, "FPR": fpr, "FNR": fnr}

# --- Dictionnaire des modèles d'embedding pour ML classifiers ---
embedding_models = {
    "DeepWalk": deepwalk_model,
    "Node2Vec": node2vec_model
}


In [ ]:
from sklearn.ensemble import RandomForestClassifier

def combine_embeddings(model, edges):
    X = []
    for u, v in edges:
        eu = model.wv[str(u)]
        ev = model.wv[str(v)]
        X.append(eu * ev)  # Hadamard product
    return np.array(X)

# Créer features
X_pos = combine_embeddings(deepwalk_model, train_edges)
X_neg = combine_embeddings(deepwalk_model, generate_negative_edges(G_train, len(train_edges)))
y_train = np.array([1]*len(X_pos) + [0]*len(X_neg))
X_train = np.vstack([X_pos, X_neg])

# Entraîner Random Forest
clf = RandomForestClassifier(n_estimators=100)
clf.fit(X_train, y_train)

# Évaluer sur validation
X_val_pos = combine_embeddings(deepwalk_model, val_edges)
X_val_neg = combine_embeddings(deepwalk_model, neg_val)
X_val = np.vstack([X_val_pos, X_val_neg])
y_val = np.array([1]*len(X_val_pos) + [0]*len(X_val_neg))

y_pred = clf.predict_proba(X_val)[:,1]
auc, ap = roc_auc_score(y_val, y_pred), average_precision_score(y_val, y_pred)


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

# ---------- Helper Functions ----------

def generate_negative_edges(G, num_samples):
    neg = []
    nodes = list(G.nodes())
    while len(neg) < num_samples:
        u, v = np.random.choice(nodes, 2, replace=False)
        if not G.has_edge(u, v):
            neg.append((u, v))
    return neg

def combine_embeddings(model, edges):
    X = []
    for u, v in edges:
        eu = model.wv[str(u)]
        ev = model.wv[str(v)]
        X.append(eu * ev)  # Hadamard product
    return np.array(X)

def compute_metrics(y_true, y_pred_scores):
    auc = roc_auc_score(y_true, y_pred_scores)
    ap = average_precision_score(y_true, y_pred_scores)
    y_pred_label = (y_pred_scores >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_label).ravel()
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    return auc, ap, fpr, fnr

def run_ml_on_embedding(model, train_edges, val_edges, G_train, ml_type="rf"):
    # --- Training edges ---
    neg_train = generate_negative_edges(G_train, len(train_edges))
    X_pos = combine_embeddings(model, train_edges)
    X_neg = combine_embeddings(model, neg_train)
    X_train = np.vstack([X_pos, X_neg])
    y_train = np.array([1]*len(X_pos) + [0]*len(X_neg))
    
    # --- ML Classifier ---
    if ml_type == "rf":
        clf = RandomForestClassifier(n_estimators=100, random_state=42)
    elif ml_type == "xgb":
        clf = XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric="logloss")
    elif ml_type == "svm":
        clf = SVC(probability=True)
    else:
        raise ValueError("Unknown ML type")
    
    clf.fit(X_train, y_train)
    
    # --- Validation edges ---
    neg_val = generate_negative_edges(G_train, len(val_edges))
    X_val = np.vstack([combine_embeddings(model, val_edges), combine_embeddings(model, neg_val)])
    y_val = np.array([1]*len(val_edges) + [0]*len(neg_val))
    
    y_pred = clf.predict_proba(X_val)[:,1]
    auc, ap, fpr, fnr = compute_metrics(y_val, y_pred)
    return auc, ap, fpr, fnr

# ---------- Run for DeepWalk and Node2Vec ----------

results_ml = {}
embedding_models = {
    "DeepWalk": deepwalk_model,
    "Node2Vec": node2vec_model
}

ml_classifiers = ["rf", "xgb", "svm"]

for emb_name, emb_model in embedding_models.items():
    for ml in ml_classifiers:
        print(f"Running {emb_name} + {ml.upper()}")
        auc, ap, fpr, fnr = run_ml_on_embedding(emb_model, train_edges, val_edges, G_train, ml_type=ml)
        results_ml[f"{emb_name}_{ml.upper()}"] = {"AUC": auc, "AP": ap, "FPR": fpr, "FNR": fnr}

# ---------- Display results ----------
results_ml


# GNN MODELS

## Prepare PyG Data

In [ ]:
import torch
from torch_geometric.utils import from_networkx, train_test_split_edges

# Convertir ton graphe NetworkX en PyG Data
data = from_networkx(G_train)

# Split edges pour link prediction
data = train_test_split_edges(data, val_ratio=0.15, test_ratio=0.15)

# Pour info
print("Nodes:", data.num_nodes, "Edges train:", data.train_pos_edge_index.size(1))


In [ ]:
from torch_geometric.nn import GCNConv, GAE, SAGEConv, VGAE

class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GCNEncoder, self).__init__()
        self.conv1 = GCNConv(in_channels, 2*out_channels)
        self.conv2 = GCNConv(2*out_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x


In [ ]:
out_dim = 64
num_features = data.num_features if hasattr(data, "num_features") else 16

# Si ton graphe n'a pas de features, on peut créer des embeddings initiaux
if not hasattr(data, 'x') or data.x is None:
    data.x = torch.eye(data.num_nodes, dtype=torch.float)

encoder = GCNEncoder(data.x.size(1), out_dim)
model = GAE(encoder)


In [ ]:
import torch.nn.functional as F

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
epochs = 50

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    z = model.encode(data.x, data.train_pos_edge_index)
    loss = model.recon_loss(z, data.train_pos_edge_index)
    loss.backward()
    optimizer.step()
    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


In [ ]:
model.eval()
with torch.no_grad():
    z = model.encode(data.x, data.train_pos_edge_index)
    pos_edge_index = data.val_pos_edge_index
    neg_edge_index = data.val_neg_edge_index
    auc, ap = model.test(z, pos_edge_index, neg_edge_index)
    print(f"GAE Val -> AUC: {auc:.4f}, AP: {ap:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

# -------------------------------
# Préparer les features
# -------------------------------
# node_embeddings : dictionnaire {node_id: vector} depuis Node2Vec ou DeepWalk
# edge_features : np.array de shape (num_edges, num_edge_features)

def prepare_data(G, edge_features_dict, node_embeddings):
    # Node features : concat de embeddings
    node_list = list(G.nodes())
    node_id_map = {node: i for i, node in enumerate(node_list)}
    x = torch.tensor([node_embeddings[str(node)] for node in node_list], dtype=torch.float)

    # Edge index
    edge_index = torch.tensor([[node_id_map[u], node_id_map[v]] for u, v in G.edges()], dtype=torch.long).t()

    # Edge features
    edge_attr = torch.tensor([edge_features_dict[(u,v)] for u,v in G.edges()], dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    return data, node_id_map

# -------------------------------
# Modèle GNN pour link prediction
# -------------------------------
class EdgeGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(EdgeGNN, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)

        # MLP pour prédire score d'edge à partir de concat(u,v)
        self.edge_mlp = nn.Sequential(
            nn.Linear(2*hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x, edge_index, edges_to_predict):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)

        # edges_to_predict : list of tuples (u,v) node indices
        u_idx = torch.tensor([u for u,v in edges_to_predict], dtype=torch.long)
        v_idx = torch.tensor([v for u,v in edges_to_predict], dtype=torch.long)

        edge_feat = torch.cat([x[u_idx], x[v_idx]], dim=1)
        score = self.edge_mlp(edge_feat).squeeze()
        return score

# -------------------------------
# Entraînement
# -------------------------------
def train_gnn(model, data, train_pos, train_neg, val_pos, val_neg, epochs=50, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    all_edges = train_pos + train_neg
    labels = torch.tensor([1]*len(train_pos) + [0]*len(train_neg), dtype=torch.float)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(data.x, data.edge_index, all_edges)
        loss = F.binary_cross_entropy(pred, labels)
        loss.backward()
        optimizer.step()
        if epoch % 10 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

    # Evaluation
    model.eval()
    val_edges = val_pos + val_neg
    y_true = [1]*len(val_pos) + [0]*len(val_neg)
    y_pred = model(data.x, data.edge_index, val_edges).detach().numpy()
    auc = roc_auc_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)
    return auc, ap

# -------------------------------
# Exemple d'utilisation
# -------------------------------
# data, node_map = prepare_data(G_train, edge_features_dict, node_embeddings)
# model = EdgeGNN(in_channels=data.num_node_features, hidden_channels=64)
# auc, ap = train_gnn(model, data, train_pos, train_neg, val_pos, val_neg)
# print("GNN Edge-aware: AUC =", auc, "AP =", ap)


In [ ]:
from torch_geometric.nn import GCNConv

class GCNLinkPredictor(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.edge_mlp = nn.Sequential(
            nn.Linear(2*hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x, edge_index, edges_to_predict):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)

        u_idx = torch.tensor([u for u,v in edges_to_predict])
        v_idx = torch.tensor([v for u,v in edges_to_predict])
        edge_feat = torch.cat([x[u_idx], x[v_idx]], dim=1)
        return self.edge_mlp(edge_feat).squeeze()


In [ ]:
from torch_geometric.nn import SAGEConv

class SAGEEdgePredictor(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.edge_mlp = nn.Sequential(
            nn.Linear(2*hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x, edge_index, edges_to_predict):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)

        u_idx = torch.tensor([u for u,v in edges_to_predict])
        v_idx = torch.tensor([v for u,v in edges_to_predict])
        edge_feat = torch.cat([x[u_idx], x[v_idx]], dim=1)
        return self.edge_mlp(edge_feat).squeeze()


In [ ]:
from torch_geometric.nn import GATConv

class GATEdgePredictor(nn.Module):
    def __init__(self, in_channels, hidden_channels, heads=2):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads)
        self.conv2 = GATConv(hidden_channels*heads, hidden_channels, heads=1)
        self.edge_mlp = nn.Sequential(
            nn.Linear(2*hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x, edge_index, edges_to_predict):
        x = F.elu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)

        u_idx = torch.tensor([u for u,v in edges_to_predict])
        v_idx = torch.tensor([v for u,v in edges_to_predict])
        edge_feat = torch.cat([x[u_idx], x[v_idx]], dim=1)
        return self.edge_mlp(edge_feat).squeeze()


In [ ]:
def nx_to_pyg(G, edge_features=None):
    # ---- Map nodes to integer indices ----
    node2idx = {n: i for i, n in enumerate(G.nodes())}
    
    # ---- Convert edges to index tensor ----
    edge_index = torch.tensor([[node2idx[u], node2idx[v]] for u, v in G.edges()], dtype=torch.long).t().contiguous()
    
    # ---- Node features: one-hot ----
    x = torch.eye(G.number_of_nodes(), dtype=torch.float)
    
    # ---- Edge features (optionnel) ----
    if edge_features is not None:
        edge_attr = torch.tensor([edge_features[(u,v)] for u,v in G.edges()], dtype=torch.float)
    else:
        edge_attr = None
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr), node2idx


In [ ]:
from torch_geometric.data import Data

# --- Convert NetworkX graph en PyG Data ---
data, node2idx = nx_to_pyg(G_train)

# --- Mapper les edges avec les indices entiers ---
train_edges_idx = [(node2idx[u], node2idx[v]) for u,v in train_edges]
val_edges_idx   = [(node2idx[u], node2idx[v]) for u,v in val_edges]
test_edges_idx  = [(node2idx[u], node2idx[v]) for u,v in test_edges]
neg_train_idx   = [(node2idx[u], node2idx[v]) for u,v in neg_train]
neg_val_idx     = [(node2idx[u], node2idx[v]) for u,v in neg_val]

# Maintenant tu peux utiliser train_edges_idx, val_edges_idx, etc. dans ton GNN


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv, GATConv
from sklearn.metrics import roc_auc_score, average_precision_score
import networkx as nx
import random
import numpy as np

# ---------------- Helper Functions ----------------
def compute_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    auc = roc_auc_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)
    fpr = np.sum((y_pred>0.5) & (y_true==0)) / max(1, np.sum(y_true==0))
    fnr = np.sum((y_pred<=0.5) & (y_true==1)) / max(1, np.sum(y_true==1))
    return auc, ap, fpr, fnr

def generate_negative_edges(G, num_samples):
    neg = []
    nodes = list(G.nodes())
    while len(neg) < num_samples:
        u, v = random.sample(nodes, 2)
        if not G.has_edge(u, v):
            neg.append((u, v))
    return neg

# ---------------- Convert NetworkX -> PyG ----------------
def nx_to_pyg(G):
    node2idx = {n: i for i, n in enumerate(G.nodes())}
    edge_index = torch.tensor([[node2idx[u], node2idx[v]] for u,v in G.edges()], dtype=torch.long).t().contiguous()
    x = torch.eye(G.number_of_nodes())  # One-hot node features
    data = Data(x=x, edge_index=edge_index)
    return data, node2idx

# ---------------- GNN Model ----------------
class GNNLinkPredictor(nn.Module):
    def __init__(self, conv_type, in_channels, hidden_channels, heads=2):
        super().__init__()
        if conv_type=="GCN":
            self.conv1 = GCNConv(in_channels, hidden_channels)
            self.conv2 = GCNConv(hidden_channels, hidden_channels)
        elif conv_type=="GraphSAGE":
            self.conv1 = SAGEConv(in_channels, hidden_channels)
            self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        elif conv_type=="GAT":
            self.conv1 = GATConv(in_channels, hidden_channels, heads=heads)
            self.conv2 = GATConv(hidden_channels*heads, hidden_channels, heads=1)
        else:
            raise ValueError("Unknown conv_type")
        
        self.edge_mlp = nn.Sequential(
            nn.Linear(2*hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x, edge_index, edges_to_predict):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        
        u_idx = torch.tensor([u for u,v in edges_to_predict], dtype=torch.long)
        v_idx = torch.tensor([v for u,v in edges_to_predict], dtype=torch.long)
        edge_feat = torch.cat([x[u_idx], x[v_idx]], dim=1)
        return self.edge_mlp(edge_feat).squeeze()

# ---------------- Train Function ----------------
def train_gnn(model, data, train_pos, train_neg, val_pos, val_neg, epochs=50, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    y_val = [1]*len(val_pos) + [0]*len(val_neg)
    edges_val = val_pos + val_neg
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        y_pred_train = model(data.x, data.edge_index, train_pos + train_neg)
        y_true_train = torch.tensor([1]*len(train_pos) + [0]*len(train_neg), dtype=torch.float)
        loss = F.binary_cross_entropy(y_pred_train, y_true_train)
        loss.backward()
        optimizer.step()
    
    model.eval()
    with torch.no_grad():
        y_pred_val = model(data.x, data.edge_index, edges_val)
        auc, ap, fpr, fnr = compute_metrics(y_val, y_pred_val.numpy())
    return auc, ap, fpr, fnr

# ---------------- Example Usage ----------------
# Charger ton graph NetworkX
# G = nx.read_graphml("your_graph.graphml")

# Shuffle et split edges
edges = list(G.edges())
random.shuffle(edges)
n = len(edges)
train_edges = edges[:int(0.7*n)]
val_edges   = edges[int(0.7*n):int(0.85*n)]
test_edges  = edges[int(0.85*n):]

G_train = nx.Graph()
G_train.add_nodes_from(G.nodes())
G_train.add_edges_from(train_edges)

# Negative edges
neg_train = generate_negative_edges(G_train, len(train_edges))
neg_val   = generate_negative_edges(G_train, len(val_edges))
neg_test  = generate_negative_edges(G_train, len(test_edges))

# Convert to PyG
data, node2idx = nx_to_pyg(G_train)

# Mapper les edges en indices
train_edges_idx = [(node2idx[u], node2idx[v]) for u,v in train_edges]
val_edges_idx   = [(node2idx[u], node2idx[v]) for u,v in val_edges]
test_edges_idx  = [(node2idx[u], node2idx[v]) for u,v in test_edges]
neg_train_idx   = [(node2idx[u], node2idx[v]) for u,v in neg_train]
neg_val_idx     = [(node2idx[u], node2idx[v]) for u,v in neg_val]
neg_test_idx    = [(node2idx[u], node2idx[v]) for u,v in neg_test]

# ---------------- Run GNN Variants ----------------
gnn_variants = ["GCN", "GraphSAGE", "GAT"]
results_gnn = {}

for conv in gnn_variants:
    print(f"\n=== Training {conv} ===")
    model = GNNLinkPredictor(conv_type=conv, in_channels=G_train.number_of_nodes(), hidden_channels=64)
    auc, ap, fpr, fnr = train_gnn(
        model,
        data,
        train_edges_idx,
        neg_train_idx,
        val_edges_idx,
        neg_val_idx,
        epochs=30,
        lr=0.01
    )
    results_gnn[conv] = {"AUC": auc, "AP": ap, "FPR": fpr, "FNR": fnr}

print("\n--- GNN Results ---")
for k,v in results_gnn.items():
    print(k, v)


## GraphSAGE Link Prediction

In [ ]:
from torch_geometric.nn import SAGEConv

class SAGE(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = SAGEConv(data.num_features, 128)
        self.conv2 = SAGEConv(128, 64)

    def encode(self, x, edge_index):
        h = self.conv1(x, edge_index).relu()
        h = self.conv2(h, edge_index)
        return h
    
    def decode(self, z, edges):
        return (z[edges[:,0]] * z[edges[:,1]]).sum(dim=1)


model = SAGE()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train loop
for epoch in range(20):
    z = model.encode(data.x, data.train_pos_edge_index)
    pos_score = model.decode(z, data.train_pos_edge_index.t())
    neg_edges = torch.tensor(generate_negative_edges(G_train, len(train_edges)))
    neg_score = model.decode(z, neg_edges)

    loss = -torch.log(torch.sigmoid(pos_score)).mean() \
           -torch.log(1 - torch.sigmoid(neg_score)).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("GraphSAGE trained.")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import from_networkx
from torch_geometric.nn import GCNConv
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def evaluate_scores(pos_scores, neg_scores):
    y_true = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
    y_score = np.concatenate([np.asarray(pos_scores), np.asarray(neg_scores)])
    return {
        "AUC": float(roc_auc_score(y_true, y_score)),
        "AP": float(average_precision_score(y_true, y_score))
    }

def edge_list_to_tensor(edge_list):
    u = [e[0] for e in edge_list]
    v = [e[1] for e in edge_list]
    return torch.tensor([u, v], dtype=torch.long)

# Ensure node indexing is int 0..N-1
num_nodes = G.number_of_nodes()
print("num_nodes:", num_nodes)


## GCN (supervised link prediction using GCN encoder + dot-product decoder)

In [ ]:
# Build PyG data and node features (degree normalized)
data_pyg = from_networkx(G_train)   # G_train is the training graph
# Create degree feature (1D) normalized
deg = np.array([G.degree(i) for i in range(num_nodes)], dtype=float)
deg = (deg - deg.mean()) / (deg.std() + 1e-9)
x = torch.tensor(deg, dtype=torch.float).unsqueeze(1)  # shape [N,1]
data_pyg.x = x

edge_index = data_pyg.edge_index.long().to(device)
data_pyg.x = data_pyg.x.to(device)

# Prepare edge tensors
train_pos = edge_list_to_tensor(train_edges).to(device)
val_pos   = edge_list_to_tensor(val_edges).to(device)
test_pos  = edge_list_to_tensor(test_edges).to(device)

train_neg = edge_list_to_tensor(train_neg).to(device)
val_neg   = edge_list_to_tensor(val_neg).to(device)
test_neg  = edge_list_to_tensor(test_neg).to(device)

class GCNEncoder(nn.Module):
    def __init__(self, in_channels, hidden=128, out_dim=64):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.conv2 = GCNConv(hidden, out_dim)
    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        z = self.conv2(h, edge_index)
        return z

def decode_dot(z, edge_index):
    # edge_index shape [2, E]
    return (z[edge_index[0]] * z[edge_index[1]]).sum(dim=1)

# Model, optimizer
model = GCNEncoder(in_channels=data_pyg.x.size(1), hidden=128, out_dim=64).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
bce = nn.BCEWithLogitsLoss()

# Training loop
epochs = 50
for epoch in range(1, epochs+1):
    model.train()
    opt.zero_grad()
    z = model(data_pyg.x, edge_index)

    pos_logits = decode_dot(z, train_pos)
    neg_logits = decode_dot(z, train_neg)

    labels = torch.cat([torch.ones(pos_logits.size(0)), torch.zeros(neg_logits.size(0))]).to(device)
    logits = torch.cat([pos_logits, neg_logits])
    loss = bce(logits, labels)
    loss.backward()
    opt.step()

    if epoch % 10 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            z = model(data_pyg.x, edge_index)
            val_pos_scores = torch.sigmoid(decode_dot(z, val_pos)).cpu().numpy()
            val_neg_scores = torch.sigmoid(decode_dot(z, val_neg)).cpu().numpy()
            metrics = evaluate_scores(val_pos_scores, val_neg_scores)
            print(f"Epoch {epoch:02d} loss={loss.item():.4f}  Val AUC={metrics['AUC']:.4f}  AP={metrics['AP']:.4f}")

# Final test evaluation
model.eval()
with torch.no_grad():
    z = model(data_pyg.x, edge_index)
    test_pos_scores = torch.sigmoid(decode_dot(z, test_pos)).cpu().numpy()
    test_neg_scores = torch.sigmoid(decode_dot(z, test_neg)).cpu().numpy()
    test_metrics = evaluate_scores(test_pos_scores, test_neg_scores)
    print("GCN Test:", test_metrics)


## GAE (Graph Autoencoder)

In [ ]:
from torch_geometric.nn import GAE, GCNConv

# Define encoder for GAE
class EncoderGAE(torch.nn.Module):
    def __init__(self, in_channels, hidden=128, out_dim=64):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.conv2 = GCNConv(hidden, out_dim)
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

# Build model
encoder = EncoderGAE(in_channels=data_pyg.x.size(1), hidden=128, out_dim=64)
gae = GAE(encoder).to(device)
opt = torch.optim.Adam(gae.parameters(), lr=0.01, weight_decay=1e-4)

# Training loop using recon_loss (PyG helper) + optional negative sampling via BCE
epochs = 60
for epoch in range(1, epochs+1):
    gae.train()
    opt.zero_grad()
    z = gae.encode(data_pyg.x, edge_index)
    loss = gae.recon_loss(z, train_pos)  # recon loss expects pos edge_index
    # Optionally add regularization (no KL for GAE)
    loss.backward()
    opt.step()

    if epoch % 10 == 0 or epoch == 1:
        gae.eval()
        with torch.no_grad():
            z = gae.encode(data_pyg.x, edge_index)
            val_pos_scores = torch.sigmoid((z[val_pos[0]] * z[val_pos[1]]).sum(dim=1)).cpu().numpy()
            val_neg_scores = torch.sigmoid((z[val_neg[0]] * z[val_neg[1]]).sum(dim=1)).cpu().numpy()
            metrics = evaluate_scores(val_pos_scores, val_neg_scores)
            print(f"Epoch {epoch:02d} GAE recon_loss={loss.item():.4f}  Val AUC={metrics['AUC']:.4f} AP={metrics['AP']:.4f}")

# Final test evaluation
gae.eval()
with torch.no_grad():
    z = gae.encode(data_pyg.x, edge_index)
    test_pos_scores = torch.sigmoid((z[test_pos[0]] * z[test_pos[1]]).sum(dim=1)).cpu().numpy()
    test_neg_scores = torch.sigmoid((z[test_neg[0]] * z[test_neg[1]]).sum(dim=1)).cpu().numpy()
    print("GAE Test:", evaluate_scores(test_pos_scores, test_neg_scores))


## VGAE (Variational Graph Autoencoder)

In [ ]:
from torch_geometric.nn import VGAE

# Encoder producing mu and logvar via two GCN layers each (we can reuse structure)
class EncoderVGAE(torch.nn.Module):
    def __init__(self, in_channels, hidden=128, out_dim=64):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.conv_mu = GCNConv(hidden, out_dim)
        self.conv_logvar = GCNConv(hidden, out_dim)
    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        return self.conv_mu(h, edge_index), self.conv_logvar(h, edge_index)

encoder = EncoderVGAE(in_channels=data_pyg.x.size(1), hidden=128, out_dim=64)
vgae = VGAE(encoder).to(device)
opt = torch.optim.Adam(vgae.parameters(), lr=0.01, weight_decay=1e-4)

# Training
epochs = 80
for epoch in range(1, epochs+1):
    vgae.train()
    opt.zero_grad()
    z = vgae.encode(data_pyg.x, edge_index)   # vgae.encode samples z internally
    # PyG VGAE has recon_loss + kl_loss helpers:
    recon_loss = vgae.recon_loss(z, train_pos)
    kl_loss = vgae.kl_loss()
    loss = recon_loss + (1.0 * kl_loss)
    loss.backward()
    opt.step()

    if epoch % 10 == 0 or epoch == 1:
        vgae.eval()
        with torch.no_grad():
            z = vgae.encode(data_pyg.x, edge_index)
            val_pos_scores = torch.sigmoid((z[val_pos[0]] * z[val_pos[1]]).sum(dim=1)).cpu().numpy()
            val_neg_scores = torch.sigmoid((z[val_neg[0]] * z[val_neg[1]]).sum(dim=1)).cpu().numpy()
            metrics = evaluate_scores(val_pos_scores, val_neg_scores)
            print(f"Epoch {epoch:02d} recon={recon_loss.item():.4f} kl={kl_loss.item():.4f} Val AUC={metrics['AUC']:.4f} AP={metrics['AP']:.4f}")

# Final test evaluation
vgae.eval()
with torch.no_grad():
    z = vgae.encode(data_pyg.x, edge_index)
    test_pos_scores = torch.sigmoid((z[test_pos[0]] * z[test_pos[1]]).sum(dim=1)).cpu().numpy()
    test_neg_scores = torch.sigmoid((z[test_neg[0]] * z[test_neg[1]]).sum(dim=1)).cpu().numpy()
    print("VGAE Test:", evaluate_scores(test_pos_scores, test_neg_scores))


In [ ]:
# -*- coding: utf-8 -*-
"""
Full pipeline link prediction:
- traditional heuristics
- shallow embeddings (node2vec/deepwalk/LINE/SVD)
- GNN (GAE/VGAE and GraphSAGE-based)
"""

import os
import random
from tqdm import tqdm
import numpy as np
import pandas as pd
import networkx as nx
import xml.etree.ElementTree as ET
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score
from sklearn.model_selection import train_test_split
from collections import defaultdict
import math

# For shallow embeddings
from gensim.models import Word2Vec

# For Node2Vec (fast random walk generator)
# pip install nodevectors  OR use custom walker below
try:
    from node2vec import Node2Vec
    node2vec_available = True
except Exception:
    node2vec_available = False

# PyG imports (GNN)
import torch
import torch.nn.functional as F
from torch_geometric.utils import from_networkx, train_test_split_edges, negative_sampling, to_undirected
from torch_geometric.nn import GCNConv, SAGEConv, GAE, VGAE

# -------------------------
# 1) Utilities: parse GraphML (manual fallback)
# -------------------------
def parse_graphml_manual(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()
    ns = {'g': 'http://graphml.graphdrawing.org/xmlns'}
    G = nx.Graph()
    keys = {}
    for key_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}key'):
        keys[key_elem.get('id')] = key_elem.get('attr.name')
    for node_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}node'):
        node_id = node_elem.get('id')
        G.add_node(node_id)
    for edge_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}edge'):
        src = edge_elem.get('source')
        tgt = edge_elem.get('target')
        attrs = {}
        for data_elem in edge_elem.findall('.//{http://graphml.graphdrawing.org/xmlns}data'):
            key_id = data_elem.get('key')
            name = keys.get(key_id)
            if name:
                try:
                    attrs[name] = float(data_elem.text)
                except:
                    attrs[name] = data_elem.text
        G.add_edge(src, tgt, **attrs)
    return G

# -------------------------
# 2) Node index mapping
# -------------------------
def relabel_nodes_to_int(G):
    mapping = {n: i for i, n in enumerate(G.nodes())}
    G_int = nx.relabel_nodes(G, mapping)
    return G_int, mapping

# -------------------------
# 3) Train/Val/Test split on edges + negative sampling
# -------------------------
def edges_train_val_test(G, val_ratio=0.05, test_ratio=0.1, seed=42):
    edges = list(G.edges())
    np.random.seed(seed)
    perm = np.random.permutation(len(edges))
    n = len(edges)
    n_test = int(n * test_ratio)
    n_val = int(n * val_ratio)
    test_idx = perm[:n_test]
    val_idx = perm[n_test:n_test+n_val]
    train_idx = perm[n_test+n_val:]
    train_edges = [edges[i] for i in train_idx]
    val_edges = [edges[i] for i in val_idx]
    test_edges = [edges[i] for i in test_idx]
    # Build train graph (remove val/test edges)
    G_train = nx.Graph()
    G_train.add_nodes_from(G.nodes(data=True))
    G_train.add_edges_from(train_edges)
    return G_train, train_edges, val_edges, test_edges

def negative_sampling_for_edges(G, positive_edges, num_neg, seed=42):
    np.random.seed(seed)
    nodes = list(G.nodes())
    node_set = set(nodes)
    neg = set()
    tries = 0
    while len(neg) < num_neg and tries < num_neg * 50:
        u = np.random.choice(nodes)
        v = np.random.choice(nodes)
        if u == v: 
            tries += 1; continue
        if (u, v) in G.edges() or (v, u) in G.edges():
            tries += 1; continue
        a = (u, v) if u < v else (v, u)
        if a in neg:
            tries += 1; continue
        neg.add(a)
    neg_list = list(neg)
    if len(neg_list) < num_neg:
        print(f"Warning: only sampled {len(neg_list)} negatives (requested {num_neg})")
    return neg_list

# -------------------------
# 4) Heuristics (NetworkX generators)
# -------------------------
def heuristic_scores(G_train, edge_list, method='adamic_adar'):
    # Returns scorelist aligned with edge_list
    scores = {}
    if method == 'jaccard':
        preds = nx.jaccard_coefficient(G_train, edge_list)
        for u, v, p in preds: scores[(u,v)] = p
    elif method == 'adamic_adar':
        preds = nx.adamic_adar_index(G_train, edge_list)
        for u, v, p in preds: scores[(u,v)] = p
    elif method == 'resource_allocation':
        preds = nx.resource_allocation_index(G_train, edge_list)
        for u, v, p in preds: scores[(u,v)] = p
    elif method == 'preferential_attachment':
        preds = nx.preferential_attachment(G_train, edge_list)
        for u, v, p in preds: scores[(u,v)] = p
    elif method == 'common_neighbors':
        # custom: number of CN
        for u, v in edge_list:
            scores[(u,v)] = len(list(nx.common_neighbors(G_train, u, v)))
    else:
        raise ValueError("Unknown heuristic")
    # return list in same order
    return [scores.get((u,v), scores.get((v,u), 0.0)) for u, v in edge_list]

# Katz approximation (truncated power series)
def katz_scores_approx(G_train, edge_list, beta=0.005, max_power=3):
    # Build adjacency in scipy sparse or numpy (dense may be huge). We'll use sparse.
    from scipy.sparse import csr_matrix, identity
    nodes = list(G_train.nodes())
    idx = {n:i for i,n in enumerate(nodes)}
    n = len(nodes)
    rows, cols = zip(*[(idx[u], idx[v]) for u,v in G_train.edges()])
    data = np.ones(len(rows))
    A = csr_matrix((data, (rows, cols)), shape=(n,n))
    A = A + A.T  # ensure undirected
    A.setdiag(0)
    # Katz = sum_{l=1..k} beta^l * A^l
    M = csr_matrix((n,n), dtype=float)
    A_pow = A.copy()
    for l in range(1, max_power+1):
        M = M + (beta**l) * A_pow
        A_pow = A_pow.dot(A)
    # score for pair u,v = M[idx[u], idx[v]]
    scores = []
    for u,v in edge_list:
        iu, iv = idx[u], idx[v]
        scores.append(float(M[iu, iv]))
    return scores

# -------------------------
# 5) Shallow embeddings
# -------------------------
# Random walk generator for DeepWalk/Node2Vec if node2vec lib not available
def generate_random_walks(G, num_walks=10, walk_length=80, p=1.0, q=1.0, seed=42):
    random.seed(seed)
    nodes = list(G.nodes())
    walks = []
    for _ in range(num_walks):
        random.shuffle(nodes)
        for node in nodes:
            walk = [str(node)]
            cur = node
            for _ in range(walk_length-1):
                neighbors = list(G.neighbors(cur))
                if len(neighbors) == 0: break
                nxt = random.choice(neighbors)
                walk.append(str(nxt))
                cur = nxt
            walks.append(walk)
    return walks

def train_word2vec_on_walks(walks, emb_dim=128, window=10, workers=4, epochs=5):
    model = Word2Vec(sentences=walks, vector_size=emb_dim, window=window, min_count=0, sg=1, workers=workers, epochs=epochs)
    return model

def get_embedding_from_word2vec(model, nodes, emb_dim=128):
    emb = np.zeros((len(nodes), emb_dim), dtype=float)
    for i, n in enumerate(nodes):
        emb[i] = model.wv[str(n)]
    return emb

# LINE or other libs not included to keep things simple. SVD:
def adjacency_svd_embedding(G_train, k=128):
    # Use sparse SVD on adjacency (TruncatedSVD on adjacency matrix rows)
    from sklearn.decomposition import TruncatedSVD
    nodes = list(G_train.nodes())
    idx = {n:i for i,n in enumerate(nodes)}
    from scipy.sparse import csr_matrix
    rows, cols = zip(*[(idx[u], idx[v]) for u,v in G_train.edges()]) if G_train.number_of_edges()>0 else ([],[])
    data = np.ones(len(rows))
    A = csr_matrix((data, (rows, cols)), shape=(len(nodes), len(nodes)))
    A = A + A.T
    svd = TruncatedSVD(n_components=k, n_iter=7, random_state=0)
    X = svd.fit_transform(A)
    return nodes, X

# Scoring by dot product or cosine
def edge_score_from_embeddings(emb_dict, edge_list, metric='dot'):
    # emb_dict: node->vector or array with mapping index
    scores=[]
    for u,v in edge_list:
        eu = emb_dict.get(str(u)) if isinstance(list(emb_dict.keys())[0], str) else emb_dict[u]
        ev = emb_dict.get(str(v)) if isinstance(list(emb_dict.keys())[0], str) else emb_dict[v]
        if eu is None or ev is None:
            scores.append(0.0); continue
        if metric=='dot':
            scores.append(float(np.dot(eu, ev)))
        elif metric=='cosine':
            denom = (np.linalg.norm(eu)*np.linalg.norm(ev))
            scores.append(float(np.dot(eu, ev)/(denom+1e-12)))
    return scores

# -------------------------
# 6) GNN: GAE / VGAE and GraphSAGE-based
# -------------------------
class EncoderGCN(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, 2*out_channels)
        self.conv2 = GCNConv(2*out_channels, out_channels)
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

class EncoderSAGE(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, 2*out_channels)
        self.conv2 = SAGEConv(2*out_channels, out_channels)
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

def train_gae(model_ga, data, optimizer, epochs=100, device='cpu'):
    model_ga.to(device)
    data = data.to(device)
    model_ga.train()
    for epoch in range(1, epochs+1):
        optimizer.zero_grad()
        z = model_ga.encode(data.x, data.train_pos_edge_index)
        loss = model_ga.recon_loss(z, data.train_pos_edge_index)
        if isinstance(model_ga, VGAE):
            loss = loss + (1 / data.num_nodes) * model_ga.kl_loss()
        loss.backward()
        optimizer.step()
    return model_ga

def evaluate_link_prediction_scores(z, pos_edges, neg_edges):
    # pos_edges and neg_edges are lists of (u,v)
    ys = []
    ypred = []
    for u,v in pos_edges:
        ys.append(1)
        ypred.append(float((z[u] * z[v]).sum()))  # dot
    for u,v in neg_edges:
        ys.append(0)
        ypred.append(float((z[u] * z[v]).sum()))
    return roc_auc_score(ys, ypred), average_precision_score(ys, ypred)

# -------------------------
# 7) Full runner
# -------------------------
def run_all(file_path, device='cpu'):
    print("Loading GraphML...")
    try:
        G = nx.read_graphml(file_path)
    except Exception as e:
        print("NetworkX read_graphml failed:", e, "-> manual parse")
        G = parse_graphml_manual(file_path)
    print("Original nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())

    # Relabel nodes to ints
    G_int, mapping = relabel_nodes_to_int(G)
    inv_mapping = {v:k for k,v in mapping.items()}
    N = G_int.number_of_nodes()

    # Train/val/test split
    G_train, train_edges, val_edges, test_edges = edges_train_val_test(G_int, val_ratio=0.05, test_ratio=0.1)
    print("Train edges:", len(train_edges), "Val:", len(val_edges), "Test:", len(test_edges))

    # Negative sampling for val/test
    val_neg = negative_sampling_for_edges(G_int, val_edges, num_neg=len(val_edges))
    test_neg = negative_sampling_for_edges(G_int, test_edges, num_neg=len(test_edges))

    # -------------------------------
    # A) Heuristics
    # -------------------------------
    heuristics = ['common_neighbors', 'jaccard', 'adamic_adar', 'resource_allocation', 'preferential_attachment']
    results = {}
    for h in heuristics:
        print("Running heuristic:", h)
        pos_scores = heuristic_scores(G_train, val_edges, method=h)
        neg_scores = heuristic_scores(G_train, val_neg, method=h)
        y_true = [1]*len(pos_scores) + [0]*len(neg_scores)
        y_score = pos_scores + neg_scores
        auc = roc_auc_score(y_true, y_score)
        ap = average_precision_score(y_true, y_score)
        results[f"heur_{h}"] = {'AUC':auc, 'AP':ap}
        print(f"  Val AUC={auc:.4f}, AP={ap:.4f}")

    # Katz (approx)
    try:
        print("Katz approx (val)...")
        pos_katz = katz_scores_approx(G_train, val_edges, beta=0.005, max_power=3)
        neg_katz = katz_scores_approx(G_train, val_neg, beta=0.005, max_power=3)
        y_true = [1]*len(pos_katz) + [0]*len(neg_katz)
        y_score = pos_katz + neg_katz
        results['heur_katz'] = {'AUC':roc_auc_score(y_true, y_score), 'AP':average_precision_score(y_true, y_score)}
    except Exception as e:
        print("Katz failed:", e)

    # -------------------------------
    # B) Shallow embeddings (DeepWalk/Node2Vec + SVD)
    # -------------------------------
    # Random walks + Word2Vec (DeepWalk)
    print("Generating random walks for DeepWalk / Node2Vec...")
    walks = generate_random_walks(G_train, num_walks=10, walk_length=40)
    w2v = train_word2vec_on_walks(walks, emb_dim=128, window=10, workers=4, epochs=5)
    nodes_list = list(G_train.nodes())
    emb = {str(node): w2v.wv[str(node)] for node in nodes_list}

    # score val
    pos_scores = edge_score_from_embeddings(emb, val_edges, metric='dot')
    neg_scores = edge_score_from_embeddings(emb, val_neg, metric='dot')
    y_true = [1]*len(pos_scores) + [0]*len(neg_scores)
    y_score = pos_scores + neg_scores
    results['deepwalk'] = {'AUC':roc_auc_score(y_true, y_score), 'AP':average_precision_score(y_true, y_score)}
    print("DeepWalk Val AUC:", results['deepwalk']['AUC'])

    # SVD
    print("SVD embedding...")
    nodes_svd, X = adjacency_svd_embedding(G_train, k=128)
    idx_map = {n:i for i,n in enumerate(nodes_svd)}
    emb_svd = {n: X[idx_map[n]] for n in nodes_svd}
    pos_scores = [float(np.dot(emb_svd[u], emb_svd[v])) for u,v in val_edges]
    neg_scores = [float(np.dot(emb_svd[u], emb_svd[v])) for u,v in val_neg]
    y_true = [1]*len(pos_scores) + [0]*len(neg_scores)
    y_score = pos_scores + neg_scores
    results['svd'] = {'AUC':roc_auc_score(y_true, y_score), 'AP':average_precision_score(y_true, y_score)}
    print("SVD Val AUC:", results['svd']['AUC'])

    # -------------------------------
    # C) GNN: prepare PyG data
    # -------------------------------
    print("Preparing PyG Data...")
    G_pyg = G_train.copy()
    # build node features: use degree or one-hot? We'll use simple degree + optional embedding if available
    degrees = np.array([G_pyg.degree(n) for n in G_pyg.nodes()], dtype=float)
    deg_norm = (degrees - degrees.mean()) / (degrees.std()+1e-12)
    x = torch.tensor(deg_norm.reshape(-1,1), dtype=torch.float)

    data = from_networkx(G_pyg)
    data.x = x
    data.num_nodes = G_pyg.number_of_nodes()
    # create positive edge_index for training (PyG expects 2xE)
    edge_index = torch.tensor(list(G_pyg.edges())).t().contiguous()
    edge_index = to_undirected(edge_index)
    data.edge_index = edge_index

    # use train_test_split_edges utility requires original graph (we already split manually)
    # we'll set data.train_pos_edge_index = train edges...
    train_pos = torch.tensor(train_edges, dtype=torch.long).t().contiguous()
    data.train_pos_edge_index = to_undirected(train_pos)

    # Create val/test edge index tensors
    val_pos = val_edges
    test_pos = test_edges
    # prepare negative edge lists as python lists (we made val_neg, test_neg)

    # GAE with GCN encoder
    in_channels = data.x.shape[1]
    hidden = 64
    out_dim = 64

    # GAE (deterministic)
    encoder = EncoderGCN(in_channels, out_dim)
    model = GAE(encoder)

    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)
    print("Training GAE...")
    model = train_gae(model, data, opt, epochs=100, device=device)

    # After training, get embeddings z (encode on train_pos_edge_index)
    model.eval()
    z = model.encode(data.x.to(device), data.train_pos_edge_index.to(device))
    z = z.detach().cpu().numpy()

    # Evaluate on val
    val_auc, val_ap = evaluate_link_prediction_scores(z, val_pos, val_neg)
    results['GAE_GCN'] = {'AUC':val_auc, 'AP':val_ap}
    print("GAE Val AUC:", val_auc, "AP:", val_ap)

    # GraphSAGE -> GAE (alternate)
    encoder_sage = EncoderSAGE(in_channels, out_dim)
    model_sage = GAE(encoder_sage)
    opt2 = torch.optim.Adam(model_sage.parameters(), lr=0.01, weight_decay=1e-5)
    model_sage = train_gae(model_sage, data, opt2, epochs=100, device=device)
    z2 = model_sage.encode(data.x.to(device), data.train_pos_edge_index.to(device)).detach().cpu().numpy()
    val_auc2, val_ap2 = evaluate_link_prediction_scores(z2, val_pos, val_neg)
    results['GAE_SAGE'] = {'AUC':val_auc2, 'AP':val_ap2}
    print("GAE SAGE Val AUC:", val_auc2)

    # -------------------------------
    # Final evaluation on test set for top methods
    # -------------------------------
    print("Final evaluation on test set for selected methods...")
    final_results = {}
    # evaluate heuristics on test
    for h in heuristics:
        print("Heuristic test:", h)
        pos_scores = heuristic_scores(G_train, test_pos, method=h)
        neg_scores = heuristic_scores(G_train, test_neg, method=h)
        y_true = [1]*len(pos_scores) + [0]*len(neg_scores)
        y_score = pos_scores + neg_scores
        final_results[f"heur_{h}"] = {'AUC':roc_auc_score(y_true, y_score), 'AP':average_precision_score(y_true, y_score)}
    # deepwalk test
    pos_scores = edge_score_from_embeddings(emb, test_pos, metric='dot')
    neg_scores = edge_score_from_embeddings(emb, test_neg, metric='dot')
    y_true = [1]*len(pos_scores) + [0]*len(neg_scores)
    final_results['deepwalk'] = {'AUC':roc_auc_score(y_true, pos_scores+neg_scores), 'AP':average_precision_score(y_true, pos_scores+neg_scores)}
    # GAE test (use z from GAE)
    test_auc, test_ap = evaluate_link_prediction_scores(z, test_pos, test_neg)
    final_results['GAE_GCN'] = {'AUC':test_auc, 'AP':test_ap}

    print("Done. Results summary (val and test):")
    print("Validation results:", results)
    print("Test results:", final_results)
    return results, final_results

# -------------------------
# 8) Launch (change file_path accordingly)
# -------------------------
if __name__ == "__main__":
    file_path = "/kaggle/input/network-intrusion-dataset/0.1M-Stratified-Multi.graphml"
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    val_res, test_res = run_all(file_path, device=device)
    # Save results
    pd.DataFrame(val_res).T.to_csv("val_results.csv")
    pd.DataFrame(test_res).T.to_csv("test_results.csv")


In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm import tqdm

# Load graphml
G = nx.read_graphml("/kaggle/input/network-intrusion-dataset/0.1M-Stratified-Multi.graphml")

# Convert node IDs to integers
G = nx.convert_node_labels_to_integers(G, label_attribute="ip")

# Split train/test edges
edges = list(G.edges())
np.random.shuffle(edges)

train_size = int(0.8 * len(edges))
train_edges = edges[:train_size]
test_edges = edges[train_size:]

G_train = nx.Graph()
G_train.add_nodes_from(G.nodes())
G_train.add_edges_from(train_edges)

# Generate negative edges
def generate_negative_edges(G, num_samples):
    negatives = []
    nodes = list(G.nodes())
    while len(negatives) < num_samples:
        u = np.random.choice(nodes)
        v = np.random.choice(nodes)
        if u != v and not G.has_edge(u, v):
            negatives.append((u, v))
    return negatives

test_neg = generate_negative_edges(G, len(test_edges))

# Evaluate heuristics
def evaluate(scores_pos, scores_neg):
    y_true = [1]*len(scores_pos) + [0]*len(scores_neg)
    y_pred = scores_pos + scores_neg
    return {
        "AUC": roc_auc_score(y_true, y_pred),
        "AP": average_precision_score(y_true, y_pred)
    }

methods = {
    "common_neighbors": lambda G,u,v: len(list(nx.common_neighbors(G,u,v))),
    "jaccard": lambda G,u,v: list(nx.jaccard_coefficient(G, [(u,v)]))[0][2],
    "adamic_adar": lambda G,u,v: list(nx.adamic_adar_index(G, [(u,v)]))[0][2],
    "resource_allocation": lambda G,u,v: list(nx.resource_allocation_index(G, [(u,v)]))[0][2],
    "preferential_attachment": lambda G,u,v: list(nx.preferential_attachment(G, [(u,v)]))[0][2],
}

results = {}

for name, func in methods.items():
    pos_scores = [func(G_train,u,v) for u,v in tqdm(test_edges)]
    neg_scores = [func(G_train,u,v) for u,v in tqdm(test_neg)]
    results[name] = evaluate(pos_scores, neg_scores)

results


In [ ]:
import numpy as np
from node2vec import Node2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# ---------------------------------------------------------
# FUNCTION: Generate negative edges
# ---------------------------------------------------------
def generate_negative_edges(G, num_samples):
    neg_edges = set()
    nodes = list(G.nodes())
    while len(neg_edges) < num_samples:
        u = np.random.choice(nodes)
        v = np.random.choice(nodes)
        if u != v and not G.has_edge(u, v):
            neg_edges.add((u, v))
    return list(neg_edges)


# ---------------------------------------------------------
# FUNCTION: AUC and AP evaluation
# ---------------------------------------------------------
def evaluate(pos_scores, neg_scores):
    y_true = [1]*len(pos_scores) + [0]*len(neg_scores)
    y_pred = list(pos_scores) + list(neg_scores)

    return {
        "AUC": roc_auc_score(y_true, y_pred),
        "AP": average_precision_score(y_true, y_pred)
    }


# ---------------------------------------------------------
# 1. Train node2vec
# ---------------------------------------------------------
node2vec = Node2Vec(
    G_train,
    dimensions=128,
    walk_length=30,
    num_walks=200,
    workers=4
)

model = node2vec.fit(window=10, min_count=1)


# ---------------------------------------------------------
# 2. Edge embedding (Hadamard)
# ---------------------------------------------------------
def edge_embedding(u, v):
    return model.wv[str(u)] * model.wv[str(v)]


# ---------------------------------------------------------
# 3. Build TRAIN dataset
# ---------------------------------------------------------
train_pos = train_edges[:20000]
train_neg = generate_negative_edges(G_train, len(train_pos))

X = []
Y = []

for u, v in train_pos:
    X.append(edge_embedding(u, v))
    Y.append(1)

for u, v in train_neg:
    X.append(edge_embedding(u, v))
    Y.append(0)

X = np.array(X)
Y = np.array(Y)


# ---------------------------------------------------------
# 4. Train the classifier
# ---------------------------------------------------------
clf = LogisticRegression(max_iter=2000)
clf.fit(X, Y)


# ---------------------------------------------------------
# 5. Evaluate NODE2VEC
# ---------------------------------------------------------
test_pos_vec = [edge_embedding(u, v) for u, v in test_edges]
test_neg_vec = [edge_embedding(u, v) for u, v in test_neg]

scores_pos = clf.predict_proba(test_pos_vec)[:, 1]
scores_neg = clf.predict_proba(test_neg_vec)[:, 1]

node2vec_results = evaluate(scores_pos, scores_neg)
node2vec_results
